# Week 2: In-Class Exercises
Due: noon Friday Sept. 11th on HuskyCT  
Submit with filename: `w2_LASTNAME.ipynb`

## Harmonic Oscillator Case Study
This is a short case-study of how to go from "raw" measurement (or simulation) data through exploratory visualization to an (almost) paper ready figure.
In this scenario, assume that you have fabricated (or simulated) 25 cantilevers.  There is some value (suggestively called "control") that varies between the cantilevers.  
To characterize them you flex them by some fixed displacement and release the tip. You then record a time series of the displacement as the vibrations damp out.

The data for these exercises comes from a `.py` script called `gendata`. 

```{literalinclude} gendata.py
:lineno-match:
``` 

In [ ]:
# standard libraries
%matplotlib inline
import matplotlib.pyplot as plt # type: ignore
# scipy libraries
import numpy as np
import pandas as pd
# custom module
import gendata as gt
# get data for N=25 cantilevers
d, time, control = gt.get_data(25) #displacement, time, control value for the corresponding experiment

<h2><span class="fa fa-flash"></span> 1. Plot a single experiment</h2>

+ Write a function to plot the displacement vs. time for a single experiment. 
    The function should take an integer between 0 and 24 for each of the cantilever experiments and plot the corresponding line. The line should be labeled with the corresponding control value.
+ Make a test figure: test your function by plotting the 7th cantilever experiment within a figure environment with axes that show the corresponding x- and y- axis labels. 

In [ ]:
def plot_one(time, index):
    """
    Takes an array of time points for the x-axis and plots the displacement of an experiment indexed by "index" from the "d" array. 
    Labels with the corresponding control value.
    """
    #(ln,) = plt.plot(...., label=....)
    #return {"raw": ln}

f,ax = plt.subplots(1)
# plot_one(..,...)
# plt.legend()
ax.set_xlabel('time [ms]')
ax.set_ylabel('displacement [mm]')


<h2><span class="fa fa-flash"></span> 2. Plot Multiple Lines </h2>

Now, let's say we want to plot multiple lines:  
+ Make a figure: use a `for` loop and your function, make a figure like the one before now showing the results of the 1st, 6th, and last experiments all on the same plot.



In [ ]:
# your code here

This is still pretty cluttered and hard to see differences in the periods, for example.  Because we know that the displacements are a relative measurement from the equilibrium position, we can try to offset the curves vertically.
+ Write a different version of your plotting command that takes in the parameter `offset`, which indicates how far in the y-direction you can plot the line. (You can just add this value to the displacement array).  
    Use the provided annotation command to replace the label of the control value to appear above each line.
+ Make a figure: plot the same 3 lines as before, this time vertically offset from one another by different amounts so as to be evenly vertically spaced on the same plot.

In [ ]:
def plot_one_offset(time, index, offset):
    """
    Takes an array of time points for the x-axis and plots the displacement of an experiment indexed by "index" from the "d" array translated in y by the input value "offset"
    Labels with the corresponding control value with annotation.
    """
    ax = plt.gca() # grabs the assumed current axes environment

    #(ln,) = plt.plot(..,...)
    control_val = 0.0 # replace
    ann = ax.annotate(
        (
            f"$C={control_val:.1f}$\n"
        ),
        # units are (axes-fraction, data)
        xy=(0.95, offset + 0.5),
        xycoords=ax.get_yaxis_transform(),
        # set the text alignment
        ha="right",
        va="bottom",
    )
    #return {"raw": ln, "annotation":ann}

fig, ax = plt.subplots()
#for ...:
    #plot_one_offset(time, index, offset)

ax.set_xlabel("time [ms]")
ax.set_ylabel("displacement + offset [mm]")

<h2><span class="fa fa-flash"></span> 3. Plotting a Fit </h2>

In a real experiment, it's common to have to compare experimental data to a fit. In `get_data`, there is a defined function `fit` that fits the displacement to the following function of time:

$$
z(t) = A e^{-\zeta\omega_0t} \sin\left(\sqrt{1 - \zeta^2}\omega_0t + \varphi\right).
$$

by fitting for the parameters: $A, \zeta, \omega, \varphi$

+ Amend your plotting function from the previous part to take the resulting `fit_vals` and use the `sample` method to plot the fit for each experiment on top of the line.
+ Add the values of $\zeta$ and $\omega$ to the annotation.
+ Make a figure of the three lines offset from one another with the corresponding fits labeled.


In [ ]:
def plot_one_offset(time, index, fitvals, offset):
    """
    Takes an array of time points for the x-axis and plots the displacement of an experiment indexed by "index" from the "d" array translated in y by the input value "offset"
    Plots the corresponding fit overlaid on top as a black line.
    Labels with the corresponding control value with an annotation.
    """
    ax = plt.gca()
    
    #(ln,) = plt.plot(...,...)
    #(fit,) = plt.plot(...,...)
    ann = ax.annotate(
        (
            f"$C={0.0:.1f}$\n" # fill in with label values
            f"$\\zeta={0.0:.2f}, \\omega={0.0:.2f}$"
        ),
        # units are (axes-fraction, data)
        xy=(0.95, offset + 0.5),
        xycoords=ax.get_yaxis_transform(),
        # set the text alignment
        ha="right",
        va="bottom",
    )
    #return {"raw": ln, "fit":fit, "annotation":ann}

fig, ax = plt.subplots()

#for ....: 
    # fit_vals = gt.fit(...)
    # plot_one_offset(time, index, fit_vals,offset)


ax.set_xlabel("time [ms]")
ax.set_ylabel("displacement + offset [mm]")

<h2><span class="fa fa-flash"></span> 4. Showing multiple panels of data </h2>

From a science point of view, we want to actually look at how the fit parameters change with the control value in aggregate across all our experiments.
For this we can bulk process the data into a dictionary and plot our correlations. But these should really go on a separate axis from each other.
+ Write two functions: `plot_zeta` and `plot_omega` that take an input axis and plot $\zeta$ and $\omega$, respectively, against the control on each axis object. Label the x- and y-axes within the function.
    Plot $\zeta$ with markers only and $\omega$ with markers and lines.
+ Test your functions on the multiple axis figure environment. 


In [ ]:
N = d.shape[0] # total number of experiments

# define dictionary to save fit results
all_fits = {"A":np.zeros([N]),
            "zeta":np.zeros([N]),
            "omega":np.zeros([N]),
            "phi":np.zeros([N]),
            "control":control}

for i, m in enumerate(d):
    this_fit = gt.fit(m,time)
    all_fits["A"][i] = this_fit.A
    all_fits["zeta"][i] = this_fit.zeta
    all_fits["omega"][i] = this_fit.omega
    all_fits["phi"][i] = this_fit.phi

def plot_zeta(ax, all_fits):
    """
    With input:
    ax: axis to plot on
    all_fits: dictionary of all parameters
    """
    ax.set_ylabel(r"$\zeta$")
    ax.set_xlabel("control ")
    ax.set_ylim(0, 0.1)
    return #ax.plot(..., ..., marker=..., color="k",linestyle=...)


def plot_omega(ax, all_fits):
    ax.set_ylabel(r"$\omega_0/2\pi$ [kHz]")
    ax.set_xlabel("control ")
    ax.set_ylim(0, 1.5)
    return #ax.plot(..., ..., marker=...,color="k",linestyle=...)

fig, (ax1, ax2) = plt.subplots(2, 1, constrained_layout=True,sharex=True)
#plot_zeta(ax1, all_fits)
#plot_omega(ax2, all_fits)


Now let's say we want to add our actual experimental lines onto another axis of this same plot. But we wrote that code assuming it would all be on one axis!
The easy thing to do here is to wrap our code for making the plot of the lines and fits into a function that can show any number of lines all on the same axis.
+ Write a function to plot several lines that uses your function from the previous part and takes an input axis and input indices.
+ Test it on the multipanel figure provided below.

In [ ]:
def plot_several(ax,idx):
    out = []
    plt.sca(ax)
    for j, index in enumerate(idx):
        fit_vals = gt.fit(d[index],time)
        arts = plot_one_offset(time,index,fit_vals,offset=4*j)

        out.append(arts)

    ax.set_xlabel("time [ms]")
    ax.set_ylabel("displacement [mm]")

    return out

plot_several(ax=plt.gca(),idx=[0,5,-1])

In [ ]:
single_col_width = 8.6 / 2.54  # single column APS figure
double_col_width = 17.8 / 2.54  # double column APS figure
fig, ax_dict = plt.subplot_mosaic(
    [["raw", "omega"], ["raw", "zeta"]], constrained_layout=True
)
fig.set_size_inches(double_col_width, double_col_width * 0.5)
indx = [0, 10, 24]
plot_several(ax_dict["raw"],indx)
plot_zeta(ax_dict["zeta"], all_fits)
plot_omega(ax_dict["omega"], all_fits)

fig.align_ylabels(list(ax_dict.values()))

#can even save this as a pdf
#fig.savefig("fig2.pdf")